# Kreo Digital — Session 07
# Unsupervised Learning & SHAP Values
**BestAgroLab | PaySim mobile-money fraud investigation**

One integrated lab: inspect transactions → discover behavioral segments → rank anomalies → explain three cases → reveal held-out fraud labels → propose a review queue.

**Learning outcomes:** prepare mixed data without leakage; choose K without labels; profile clusters; compare density and isolation methods; explain an exact anomaly score with SHAP; evaluate under limited review capacity.

**Time:** 3–4 hours plus report. Prerequisites: Python, pandas, train/test separation and classification metrics. Working reference code is provided; complete the embedded exercises and written case notes.

## 1. Access and reproducibility
Dataset: [Synthetic Financial Datasets For Fraud Detection — PaySim](https://www.kaggle.com/datasets/ealaxi/paysim1).

Download and extract the PaySim CSV into `session-07/data/`. This is a public Kaggle dataset, not the previous invitation-only competition. Check its usage terms. The actual published filename is `PS_20174392719_1491204439457_log.csv`; the loader discovers a single matching CSV, or accepts an explicit `KREO_DATA_PATH` environment variable.

```bash
python -m pip install -r requirements.txt
kaggle datasets download -d ealaxi/paysim1 -p data --unzip
jupyter lab
```

Start Jupyter from `session-07` and run all cells in order. Default MAX_ROWS=50,000 is a label-independent uniform sample across the complete CSV, not the first 50,000 transactions. Initial loading scans the full file in chunks. No synthetic fallback is loaded. Do not commit credentials, raw data or row-level outputs.

In [ ]:
from pathlib import Path
import os, json, hashlib, platform, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (silhouette_score, average_precision_score, roc_auc_score,
                             precision_recall_curve, confusion_matrix, accuracy_score)
SEED = 42
rng = np.random.default_rng(SEED)
candidates = sorted(Path('data').glob('PS_*_log.csv'))
configured_path = os.environ.get('KREO_DATA_PATH')
DATA_PATH = Path(configured_path) if configured_path else (candidates[0] if len(candidates)==1 else Path('data/PS_20174392719_1491204439457_log.csv'))
MAX_ROWS = 50000  # None uses all ~6.36 million rows; requires substantially more RAM
REVIEW_BUDGET = 100
RUN_HYBRID = True  # optional extension, still inside this notebook
OUTPUT = Path('outputs'); OUTPUT.mkdir(exist_ok=True)
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True})
versions = {p: importlib.metadata.version(p) for p in
            ['numpy','pandas','scikit-learn','matplotlib','shap']}
print(platform.python_version(), versions)

## 2. Verify the PaySim schema and scan the full file
Published fields: `step`, `type`, `amount`, `nameOrig`, `oldbalanceOrg`, `newbalanceOrig`, `nameDest`, `oldbalanceDest`, `newbalanceDest`, `isFraud`, `isFlaggedFraud`.

One step represents one simulated hour. Types are CASH_IN, CASH_OUT, DEBIT, PAYMENT and TRANSFER. Account identifiers are not behavioral distances. `isFlaggedFraud` is an existing business-rule flag, not the truth label.

**Important simulator caveat:** the provider warns that fraudulent transactions are canceled and balance fields can reflect that treatment. We exclude **all four balance columns** from the predictive baseline rather than derive balance-error features that may reveal the simulator's handling of fraud. Zero balances are not automatically missing. There are no bank-account missing-value sentinel rules to reuse here.

The loader checks every chunk, records source hash and raw row count, and keeps a reproducible uniform random-priority sample without using labels. Row offsets provide a transaction reference, not a model feature.

In [ ]:
NUMERIC = ['step','amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','isFraud','isFlaggedFraud']
EXPECTED = set(NUMERIC + ['type','nameOrig','nameDest'])
TYPES = {'CASH_IN','CASH_OUT','DEBIT','PAYMENT','TRANSFER'}

def validate_frame(frame):
    missing, extra = EXPECTED-set(frame.columns), set(frame.columns)-EXPECTED
    if missing or extra:
        raise ValueError(f'PaySim schema mismatch. Missing={sorted(missing)}; extra={sorted(extra)}.')
    if frame.empty or frame.isna().any().any():
        raise ValueError('Expected nonempty complete PaySim rows; investigate missing values.')
    for c in NUMERIC:
        frame[c] = pd.to_numeric(frame[c], errors='raise')
    if not np.isfinite(frame[NUMERIC].to_numpy(dtype=float)).all():
        raise ValueError('Nonfinite numeric value.')
    for c in ['isFraud','isFlaggedFraud']:
        if not frame[c].isin([0,1]).all(): raise ValueError(f'{c} must be binary.')
    if not frame.type.isin(TYPES).all(): raise ValueError('Unexpected transaction type.')
    if (frame.amount < 0).any() or (frame.step < 1).any() or (frame.step % 1 != 0).any():
        raise ValueError('Expected nonnegative amount and positive integer hour steps.')
    return frame

if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Missing {DATA_PATH.resolve()}. Download and extract ealaxi/paysim1 or set KREO_DATA_PATH.')
if MAX_ROWS is not None and MAX_ROWS < 1000:
    raise ValueError('Use at least 1000 sampled rows for this lab.')
sampling_rng = np.random.default_rng(SEED)
retained, priorities, raw_rows = None, None, 0
for chunk in pd.read_csv(DATA_PATH,chunksize=200000):
    chunk = validate_frame(chunk)
    chunk.index = np.arange(raw_rows,raw_rows+len(chunk))
    raw_rows += len(chunk)
    if MAX_ROWS is None:
        retained = chunk if retained is None else pd.concat([retained,chunk])
    else:
        keys = sampling_rng.random(len(chunk))
        retained = chunk if retained is None else pd.concat([retained,chunk])
        priorities = keys if priorities is None else np.concatenate([priorities,keys])
        if len(retained) > MAX_ROWS:
            keep = np.argpartition(priorities,MAX_ROWS-1)[:MAX_ROWS]
            retained = retained.iloc[keep]
            priorities = priorities[keep]
df = retained.sort_index().copy()
assert df.index.is_unique
hasher=hashlib.sha256()
with DATA_PATH.open('rb') as handle:
    for block in iter(lambda:handle.read(1024*1024),b''): hasher.update(block)
manifest = {'dataset':'ealaxi/paysim1','source':str(DATA_PATH),'sha256':hasher.hexdigest(),
            'raw_rows':raw_rows,'used_rows':len(df),'columns':list(df.columns),
            'versions':versions,'seed':SEED,'sampling':'uniform random-priority across complete CSV'}
(OUTPUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Raw / sampled rows:',raw_rows,len(df)); display(df.drop(columns=['isFraud','isFlaggedFraud']).head())

## 3. Temporal split and feature availability
Reserve transactions above the 80th percentile of sampled step values as the holdout using a label-free cutoff. All rows in the same step stay together; never split within an hour. Record the cutoff in the manifest.

Baseline features: `log1p(amount)`, transaction `type`, and cyclical hour-of-day (`sin`, `cos`). No account identifiers, absolute time step, labels, rule flag, balances or balance discrepancies enter the models. This conservative feature set may have weak fraud enrichment; honest evaluation matters more than manufacturing a strong result.

**Exercise 1:** Write an availability table for all 11 raw columns. Explain why post-transaction balances can be problematic even when explaining already-completed transactions. Would a pre-authorization model have the same available fields?

In [ ]:
cutoff = int(df.step.quantile(.8))
train = df.loc[df.step <= cutoff].copy()
test = df.loc[df.step > cutoff].copy()
assert len(train)>=200 and len(test)>=200, 'Insufficient temporal coverage.'

def make_features(raw):
    hour = (raw.step-1) % 24
    return pd.DataFrame({'log_amount':np.log1p(raw.amount), 'type':raw.type,
                         'hour_sin':np.sin(2*np.pi*hour/24),
                         'hour_cos':np.cos(2*np.pi*hour/24)},index=raw.index)

X_train,X_test = make_features(train),make_features(test)
y_test = test.isFraud.astype(int).to_numpy()  # sealed until section 10
assert set(X_train)=={'log_amount','type','hour_sin','hour_cos'}
manifest['time_cutoff_step']=cutoff
(OUTPUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Cutoff hour:',cutoff,'Fit / holdout:',X_train.shape,X_test.shape)

In [ ]:
audit = pd.DataFrame({'dtype':X_train.dtypes.astype(str),'missing_fraction':X_train.isna().mean(),
                      'distinct':X_train.nunique()})
display(audit); display(train[['amount','step']].describe().T)
audit.to_csv(OUTPUT/'training_feature_audit.csv')
fig,ax=plt.subplots(1,3,figsize=(14,4))
ax[0].hist(train.amount,bins=60); ax[0].set_title('Raw amount')
ax[1].hist(X_train.log_amount,bins=50); ax[1].set_title('log1p amount')
train.type.value_counts().plot.bar(ax=ax[2]); ax[2].set_title('Transaction type')
plt.tight_layout(); plt.show()
print('Duplicate behavioral feature rows:',X_train.duplicated().sum())

**Exercise 2 — EDA:** Describe three patterns using the training data only. Quantify missingness and skew; inspect transaction-type proportions and the difference between raw and log amounts. Which outliers might be legitimate? Would removing duplicate feature rows accidentally remove distinct applicants? Do not inspect test labels to choose features.

## 4. Training-only mixed-type preprocessing
Median imputation, robust scaling, and one-hot encoding are fitted only on training rows. Dense float32 output is practical for this low-cardinality dataset; an explicit memory guard prevents accidental high-cardinality expansion. RobustScaler does not make Euclidean distances automatically meaningful.

In [ ]:
num = X_train.select_dtypes(include='number').columns.tolist()
cat = [c for c in X_train if c not in num]
assert not X_train.isna().all().any(), 'All-null training feature requires a documented decision.'
expected_width = len(num) + sum(X_train[c].nunique(dropna=True) for c in cat)
assert (len(X_train)+len(X_test))*expected_width*8 < 1_500_000_000, 'Reduce MAX_ROWS or use sparse encoding.'
prep = ColumnTransformer([
    ('num', Pipeline([('impute',SimpleImputer(strategy='median')),
                      ('scale',RobustScaler())]), num),
    ('cat', Pipeline([('impute',SimpleImputer(strategy='most_frequent')),
                      ('encode',OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat)
])
Z_train = prep.fit_transform(X_train).astype('float32')
Z_test = prep.transform(X_test).astype('float32')
feature_names = prep.get_feature_names_out()
assert np.isfinite(Z_train).all() and np.isfinite(Z_test).all()
print('Encoded dimensions:', Z_train.shape, Z_test.shape)

## 5. K-Means: choose K without fraud labels
Compare K=2,3,5,8 using inertia and a reproducible silhouette sample. Inertia always tends to improve with K; it is not enough on its own. We select the best sampled silhouette as a baseline rule fixed before label reveal.

In [ ]:
models, diagnostics = {}, []
for k in [2,3,5,8]:
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED)
    labels = model.fit_predict(Z_train)
    sil = silhouette_score(Z_train, labels, sample_size=min(1500,len(Z_train)), random_state=SEED)
    models[k] = model
    diagnostics.append({'k':k, 'inertia':model.inertia_, 'silhouette':sil})
selection = pd.DataFrame(diagnostics).set_index('k')
display(selection)
selection.plot(subplots=True, marker='o', figsize=(8,5)); plt.tight_layout(); plt.show()
K = int(selection.silhouette.idxmax()); km = models[K]
train_cluster, test_cluster = km.labels_, km.predict(Z_test)
print('Selected K:', K)
profile_features = ['amount','step']
profiles = train.assign(cluster=train_cluster).groupby('cluster')[profile_features].median()
profiles.insert(0,'count',pd.Series(train_cluster).value_counts().sort_index())
profiles.insert(1,'share',profiles['count']/len(X_train))
profiles['median_hour'] = train.assign(cluster=train_cluster,hour=(train.step-1)%24).groupby('cluster').hour.median()
display(profiles)
profiles.to_csv(OUTPUT/'training_cluster_profiles.csv')
for c in cat:
    print(c); display(pd.crosstab(train_cluster, X_train[c].to_numpy(), normalize='index').round(3))

**Exercise 3:** Give each cluster a neutral name based on amount and hour patterns and transaction type. Compare K=3 and K=5 and a different seed. Explain why a tiny cluster is not automatically fraudulent. Deliver a profile table and a short stability discussion.

## 6. PCA and bounded DBSCAN experiment
Fit PCA on training features only; plot a fixed random sample. The model still uses the full encoded representation. DBSCAN runs on at most 2,000 training rows after a separate 10-component PCA. Its noise flags apply only to that subset: DBSCAN has no native `predict` for the holdout.

In [ ]:
pca = PCA(n_components=2, random_state=SEED).fit(Z_train)
ix = rng.choice(len(Z_train), min(3000,len(Z_train)), replace=False)
coords = pca.transform(Z_train[ix])
plt.scatter(coords[:,0],coords[:,1],c=train_cluster[ix],s=8,cmap='tab10',alpha=.5)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(f'K-Means; PCA variance {pca.explained_variance_ratio_.sum():.1%}')
plt.show()
db_ix = rng.choice(len(Z_train), min(2000,len(Z_train)), replace=False)
reducer = PCA(n_components=min(10,Z_train.shape[1]),random_state=SEED)
Zd = reducer.fit_transform(Z_train[db_ix])
db_rows=[]
for eps in [.5,1.,2.,4.]:
    lab = DBSCAN(eps=eps,min_samples=20).fit_predict(Zd)
    db_rows.append({'eps':eps,'clusters':len(set(lab)-{-1}),'noise_share':np.mean(lab==-1)})
display(pd.DataFrame(db_rows))
db_labels = DBSCAN(eps=2.,min_samples=20).fit_predict(Zd)
plt.scatter(Zd[:,0],Zd[:,1],c=db_labels,s=10,cmap='tab10'); plt.title('DBSCAN training subset (−1 = noise)'); plt.show()

**Exercise 4:** Explain all-noise and one-cluster outcomes. How do PCA scaling, `eps`, and sample density change the answer? Do not call DBSCAN noise a fraud prediction. Deliver a parameter/noise table.

## 7. Isolation Forest and a frozen review policy
Higher `-score_samples` means more anomalous. It is not a fraud probability. Fit with no labels. Freeze a capacity-based queue and hybrid weights now, before fraud evaluation. The top-100 policy always fills available capacity, even if no meaningful anomaly exists.

In [ ]:
iso = IsolationForest(n_estimators=150,max_samples=256,contamination='auto',random_state=SEED,n_jobs=1)
iso.fit(Z_train)
score_train = -iso.score_samples(Z_train)
score_test = -iso.score_samples(Z_test)
distance_train = np.linalg.norm(Z_train-km.cluster_centers_[train_cluster],axis=1)
distance_test = np.linalg.norm(Z_test-km.cluster_centers_[test_cluster],axis=1)

def training_percentile(reference, values):
    return np.searchsorted(np.sort(reference),values,side='right') / len(reference)

scores = {'Isolation Forest': score_test, 'Centroid distance':distance_test}
if RUN_HYBRID:
    # Fixed unlabeled rank fusion; NOT supervised probabilities or calibrated risk.
    scores['Hybrid 70/30'] = .7*training_percentile(score_train,score_test) + .3*training_percentile(distance_train,distance_test)
queue_positions = np.argsort(-score_test,kind='stable')[:min(REVIEW_BUDGET,len(test))]
plt.hist(score_train,bins=50,alpha=.6,label='Training'); plt.hist(score_test,bins=50,alpha=.6,label='Holdout')
plt.xlabel('Anomaly score'); plt.ylabel('Applications'); plt.legend(); plt.show()

## 8. SHAP: explain the exact anomaly output
We use permutation SHAP on a small encoded background, explaining exactly **`-iso.score_samples`**. This avoids confusing tree path-length explanations with the nonlinear anomaly score. Positive values increase anomaly relative to the background. One-hot features are explained individually; masked combinations can be unrealistic, and correlated features can share attribution arbitrarily.

Explain the top three queued cases. Save three waterfall plots and check that base value plus contributions reconstructs the defined output. Background and evaluation limits are fixed for reproducibility; this is approximate, not a causal decomposition.

In [ ]:
import shap
bg_ix = np.random.default_rng(SEED).choice(len(Z_train),min(30,len(Z_train)),replace=False)
background = Z_train[bg_ix]
case_positions = queue_positions[:3]
case_matrix = Z_test[case_positions]
def anomaly_fn(a):
    return -iso.score_samples(np.asarray(a,dtype=np.float32))
explainer = shap.Explainer(anomaly_fn,background,algorithm='permutation',
                           feature_names=list(feature_names),seed=SEED)
explanations = explainer(case_matrix,max_evals=4*Z_train.shape[1]+1)
reconstructed = explanations.base_values + explanations.values.sum(axis=1)
np.testing.assert_allclose(reconstructed,anomaly_fn(case_matrix),atol=1e-6)
case_notes=[]
for j,pos in enumerate(case_positions):
    shap.plots.waterfall(explanations[j],max_display=12,show=False)
    plt.gcf().savefig(OUTPUT/f'case_{j+1}_waterfall.png',bbox_inches='tight'); plt.show()
    top = np.argsort(-np.abs(explanations.values[j]))[:5]
    note = {'application_id':int(test.index[pos]),'anomaly_score':float(score_test[pos]),
            'cluster':int(test_cluster[pos]),
            'top_contributions':{str(feature_names[t]):float(explanations.values[j,t]) for t in top}}
    case_notes.append(note)
    display(pd.DataFrame({'feature':feature_names[top],'contribution':explanations.values[j,top]}))
(OUTPUT/'case_attributions.json').write_text(json.dumps(case_notes,indent=2))
shap.plots.bar(explanations,show=False); plt.title('Mean |SHAP| for three flagged cases only'); plt.show()

**Exercise 5 — three investigation notes:** For each waterfall, describe two positive and one negative contribution, compare with raw feature values, and propose one follow-up verification. Repeat with a second training background. Does the story change? The three-case SHAP bar chart is **not global importance for the population**. Do not infer criminal intent from an attribution.

## 9. Cluster surrogate: approximation, not direct K-Means explanation
A Random Forest learns the training cluster IDs, not fraud. We split training rows internally to report independent surrogate agreement; the final fraud holdout remains untouched. Explain one cluster probability using a scalar callable to avoid multiclass SHAP shape ambiguity.

In [ ]:
a,b = train_test_split(np.arange(len(Z_train)),test_size=.2,random_state=SEED)
surrogate = RandomForestClassifier(n_estimators=80,max_depth=12,random_state=SEED,n_jobs=1)
surrogate.fit(Z_train[a],train_cluster[a])
print('Internal held-out cluster agreement:',accuracy_score(train_cluster[b],surrogate.predict(Z_train[b])))
chosen_cluster = int(train_cluster[b[0]])
class_index = int(np.flatnonzero(surrogate.classes_==chosen_cluster)[0])
def cluster_probability(a):
    return surrogate.predict_proba(np.asarray(a,dtype=np.float32))[:,class_index]
surrogate_explainer = shap.Explainer(cluster_probability,background,algorithm='permutation',
                                     feature_names=list(feature_names),seed=SEED)
sv = surrogate_explainer(Z_train[b[:1]],max_evals=2*Z_train.shape[1]+1)
np.testing.assert_allclose(sv.base_values+sv.values.sum(axis=1),cluster_probability(Z_train[b[:1]]),atol=1e-6)
shap.plots.waterfall(sv[0],max_display=10,show=False); plt.title(f'Surrogate probability of cluster {chosen_cluster}'); plt.show()

**Question:** If agreement is low, why is the surrogate explanation unreliable? Even with high agreement, why does it explain neither the K-Means objective nor fraud risk?

## 10. Reveal held-out fraud labels once
All choices above are label-free. Now evaluate the frozen scores. AP's reference is prevalence; random precision at K is prevalence and expected random recall at K is K/N. ROC-AUC complements AP but can look optimistic for rare events. No useful lift is a valid finding.

In [ ]:
if np.unique(y_test).size != 2:
    raise ValueError('Holdout needs both outcomes; increase MAX_ROWS or inspect temporal coverage; never resample until results look good.')
base_rate = y_test.mean(); budget = min(REVIEW_BUDGET,len(y_test))
def evaluate(score):
    order = np.argsort(-score,kind='stable'); top=y_test[order[:budget]]
    return {'AP':average_precision_score(y_test,score),'ROC_AUC':roc_auc_score(y_test,score),
            'precision_at_k':top.mean(),'recall_at_k':top.sum()/y_test.sum(),
            'lift_at_k':top.mean()/base_rate,'frauds_found':int(top.sum()),'k':budget}
metrics = pd.DataFrame({name:evaluate(score) for name,score in scores.items()}).T
print('Holdout prevalence:',base_rate,'Expected random recall:',budget/len(y_test))
display(metrics); metrics.to_csv(OUTPUT/'holdout_metrics.csv')
for name,score in scores.items():
    precision,recall,_ = precision_recall_curve(y_test,score)
    plt.plot(recall,precision,label=name)
plt.axhline(base_rate,ls='--',color='gray',label='Prevalence'); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend(); plt.show()
enrichment = pd.DataFrame({'cluster':test_cluster,'fraud':y_test}).groupby('cluster').fraud.agg(['size','sum','mean'])
enrichment['lift'] = enrichment['mean']/base_rate
enrichment['fraud_capture_share'] = enrichment['sum']/y_test.sum()
display(enrichment); enrichment.to_csv(OUTPUT/'holdout_cluster_enrichment.csv')
pred = np.zeros(len(y_test),dtype=int); pred[queue_positions]=1
cm = confusion_matrix(y_test,pred,labels=[0,1])
display(pd.DataFrame(cm,index=['actual legit','actual fraud'],columns=['not reviewed','reviewed']))

## 11. Investigation queue and final deliverables
The queue is ordered by Isolation Forest, not by revealed fraud labels. Labels are retained only in a separate evaluation table. SHAP contributions are populated only for the three explained cases; never imply every queue entry was explained.

In [ ]:
queue = pd.DataFrame({'application_id':test.index.to_numpy()[queue_positions],
                      'rank':np.arange(1,len(queue_positions)+1),
                      'anomaly_score':score_test[queue_positions],
                      'cluster':test_cluster[queue_positions]})
contributions = {n['application_id']:json.dumps(n['top_contributions']) for n in case_notes}
queue['shap_top_features'] = queue.application_id.map(contributions).fillna('Not computed')
queue['analyst_interpretation'] = ''
queue['follow_up_verification'] = ''
queue.to_csv(OUTPUT/'investigation_queue.csv',index=False)
display(queue.head(10))
print('Generated files:', sorted(p.name for p in OUTPUT.iterdir()))

## 12. Student submission and critical thinking
Submit **this one executed notebook**, its `outputs/` directory, and a 1–2 page conclusion (a Markdown cell is sufficient). Include:

1. Feature-availability audit and three EDA findings.
2. K diagnostics, neutral cluster profiles, PCA plot, DBSCAN experiment and stability notes.
3. Frozen-score metric table, base-rate comparisons and capacity-based confusion matrix.
4. Three waterfall plots and three completed investigation notes.
5. Completed review queue; explain which rows do not yet have SHAP values.
6. Optional hybrid comparison: were weights fixed before evaluation? Does mixing weak signals help? If changing weights after this reveal, call the result exploratory and obtain a new holdout before claiming improvement.

### Critical-thinking questions
- Can fraud occur in the largest, most ordinary cluster? Can an honest applicant be the strongest anomaly?
- Does temporal drift explain poor enrichment? How would a random split change the question?
- Why can AP improve while recall at a fixed review budget gets worse?
- Why is anomaly rank not a fraud probability? What labeled data would calibration need?
- PaySim has no reliable demographic attributes. Why can we not claim demographic fairness from this dataset? Could transaction types or amounts still produce unequal review burdens? Propose group-level review-rate and false-positive checks, minimum sample sizes, and a policy for discussing uncertainty.
- What changes if no fraud labels ever arrive? Which conclusions become impossible?
- How would label delay, duplicated applications, and feature availability affect production use?
- Why does SHAP not fix leakage, prove causation, or justify automatic rejection?
- How would you monitor missingness, feature drift, cluster shares, review yield and analyst feedback over time?

### Assessment rubric
| Criterion | Weight |
|---|---:|
| Data quality and leakage controls | 20% |
| Clustering and anomaly modeling | 25% |
| Cluster interpretation | 15% |
| SHAP reasoning and three cases | 15% |
| Evaluation and review prioritization | 15% |
| Reproducibility and written reasoning | 10% |

**Use limitations:** BAF is synthetic and imbalanced. Patterns may not generalize. Use this lab for education and investigation prioritization with human review.

### Reference alignment
Adapted from the supplied Session 07 outline. Its original restricted bank-account dataset is replaced with PaySim at your request. EDA, K-Means, DBSCAN, PCA, Isolation Forest, direct-score SHAP, surrogate caveats, evaluation and three-case investigation remain one coherent lab. No graph links or demographic fields are invented.

### Sources
- PaySim dataset and provider caveats: https://www.kaggle.com/datasets/ealaxi/paysim1
- Simulator source: https://github.com/EdgarLopezPhD/PaySim
- K-Means: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html
- Isolation Forest score convention: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
- SHAP permutation explainer: https://shap.readthedocs.io/en/latest/generated/shap.PermutationExplainer.html